# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/suha-2004/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [13]:
!git clone https://github.com/suha-2004/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 199, done.
remote: Counting objects: 100% (199/199), done.
remote: Compressing objects: 100% (153/153), done.
remote: Total 199 (delta 89), reused 102 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (199/199), 3.04 MiB | 17.22 MiB/s, done.
Resolving deltas: 100% (89/89), done.


In [14]:
%cd flyrank-ml-internship

/content/flyrank-ml-internship/flyrank-ml-internship


In [15]:
import pandas as pd
import numpy as np

df = pd.read_csv("./data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)

Dataset shape: (30000, 44)


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*
### Method choice

I use a Random Forest classifier for the modeling lane.

Random Forest is suitable because the task involves multiple content and search-performance signals and may contain nonlinear relationships between the predictors and the target. It also provides feature importance information that can help explain which signals contribute to the model's decisions.

The model is used as decision-support for identifying content opportunities rather than as a claim of causal ranking improvement.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.ensemble import RandomForestClassifier

print("Model:", "RandomForestClassifier")
print("Number of trees:", 300)

Model: RandomForestClassifier
Number of trees: 300


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*
### Split design

The evaluation uses a client-aware split so that client groups are separated between training and testing. This reduces the risk that the model is evaluated on the same client patterns that it saw during training.

The split is intended to provide a more honest estimate of how the model behaves on unseen client groups. The test set is kept separate from model training and is used only for evaluation.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit

group_col = "client_id"

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(df, groups=df[group_col])
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

print("Train clients:", train_df[group_col].nunique())
print("Test clients:", test_df[group_col].nunique())

print(
    "Client overlap:",
    len(
        set(train_df[group_col])
        .intersection(set(test_df[group_col]))
    )
)

Train rows: 19166
Test rows: 10834
Train clients: 22
Test clients: 10
Client overlap: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Features used in the modeling lane

feature_cols = [
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "engagement_rate"
]

# Create the target from the observed recent-period change
train_df["is_declining"] = (
    train_df["impressions_last_30d"]
    < 0.8 * train_df["impressions_prev_30d"]
).astype(int)

test_df["is_declining"] = (
    test_df["impressions_last_30d"]
    < 0.8 * test_df["impressions_prev_30d"]
).astype(int)

X_train = train_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()

y_train = train_df["is_declining"]
y_test = test_df["is_declining"]

# Numeric conversion and missing-value handling
X_train = X_train.apply(pd.to_numeric, errors="coerce")
X_test = X_test.apply(pd.to_numeric, errors="coerce")

X_train = X_train.fillna(X_train.median())
X_test = X_test.fillna(X_train.median())

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("Positive rate in train:", y_train.mean())
print("Positive rate in test:", y_test.mean())

X_train: (19166, 7)
X_test: (10834, 7)
Positive rate in train: 0.5322445998121674
Positive rate in test: 0.5594424958464095


In [19]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, y_prob)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("ROC AUC:", round(roc_auc, 3))
print("Precision:", round(precision, 3))
print("Recall:", round(recall, 3))
print("F1:", round(f1, 3))

ROC AUC: 0.651
Precision: 0.622
Recall: 0.787
F1: 0.695


In [20]:
# Rank test observations by predicted probability

ranking = test_df.copy()
ranking["model_score"] = y_prob

ranking = ranking.sort_values(
    "model_score",
    ascending=False
)

def precision_at_k(data, k):
    top_k = data.head(k)
    return top_k["is_declining"].mean()

for k in [20, 50, 100]:
    print(
        f"Precision@{k}:",
        round(precision_at_k(ranking, k), 3)
    )

Precision@20: 0.8
Precision@50: 0.8
Precision@100: 0.77


In [21]:
results = pd.DataFrame({
    "Metric": [
        "ROC AUC",
        "Precision",
        "Recall",
        "F1",
        "Precision@20",
        "Precision@50",
        "Precision@100"
    ],
    "Random Forest": [
        roc_auc,
        precision,
        recall,
        f1,
        precision_at_k(ranking, 20),
        precision_at_k(ranking, 50),
        precision_at_k(ranking, 100)
    ]
})

results

,Metric,Random Forest
0,ROC AUC,0.651392
1,Precision,0.622486
2,Recall,0.786504
3,F1,0.694949
4,Precision@20,0.800000
5,Precision@50,0.800000
6,Precision@100,0.770000


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*
### Error analysis and interpretation

The model makes both false-positive and false-negative predictions, showing that the available signals do not perfectly separate declining and non-declining content. Feature importance provides a directional view of which predictors the Random Forest relies on most heavily. These results should be interpreted as decision-support evidence rather than as causal explanations of search performance.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Error analysis

errors = test_df.copy()

errors["actual"] = y_test.values
errors["predicted"] = y_pred
errors["model_score"] = y_prob

false_positives = errors[
    (errors["actual"] == 0) &
    (errors["predicted"] == 1)
]

false_negatives = errors[
    (errors["actual"] == 1) &
    (errors["predicted"] == 0)
]

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

False positives: 2891
False negatives: 1294


In [23]:
# Feature importance

feature_importance = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": model.feature_importances_
}).sort_values(
    "Importance",
    ascending=False
)

feature_importance

,Feature,Importance
0,impressions_prev_30d,0.415841
3,content_age_days,0.207109
5,ctr,0.117503
2,sessions_prev_30d,0.099563
6,engagement_rate,0.055425
4,days_since_last_update,0.052787
1,clicks_prev_30d,0.051771


In [24]:
print("ML-08 SELF-CHECK")
print("================")

print("Dataset shape:", df.shape)
print("Number of features:", len(feature_cols))

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

print("Client overlap:",
      len(
          set(train_df["client_id"])
          .intersection(set(test_df["client_id"]))
      ))

print("Model trained:", hasattr(model, "feature_importances_"))
print("Evaluation completed:", len(results) == 7)
print("Error analysis completed:", True)

ML-08 SELF-CHECK
Dataset shape: (30000, 44)
Number of features: 7
Train rows: 19166
Test rows: 10834
Client overlap: 0
Model trained: True
Evaluation completed: True
Error analysis completed: True


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.